In [1]:
!pip install torch tokenizers -q

In [99]:
import torch

dim = 8
seq_len = 16
base = 10000.0
device = torch.device("cpu")
dtype = torch.float16

x = torch.randn(seq_len, dim, dtype=dtype, device=device)
positions = torch.arange(seq_len, device=device, dtype=dtype)
pair_idx = torch.arange(dim // 2, device=device, dtype=dtype)

# omega_i = base^(-2i/d)
inv_freq = base ** (-2.0 * pair_idx / dim)   # shape: [dim//2]
# inv_freq2 = 1.0 / (base ** (torch.arange(0, dim, 2).type(dtype) / dim))

thetas = positions[:, None] * inv_freq[None, :]

cos_t = torch.cos(thetas)   # [seq_len, dim//2]
sin_t = torch.sin(thetas)   # [seq_len, dim//2]

x_even = x[:, 0::2]   # [seq_len, dim//2]
x_odd = x[:, 1::2]    # [seq_len, dim//2]

cos = thetas.cos()
sin = thetas.sin()

x1 = x[..., : dim // 2]
x2 = x[..., dim // 2 :]


cos_t.shape, x1.shape

(torch.Size([16, 4]), torch.Size([16, 4]))

In [48]:
import torch


def rope_one_vector_python(x, position, base=10000.0):
    import math

    dim = len(x)
    out = [0.0] * dim

    for i in range(dim // 2):
        omega_i = base ** (-2.0 * i / dim)
        theta = position * omega_i
        c = math.cos(theta)
        s = math.sin(theta)

        x_even = x[2 * i]
        x_odd = x[2 * i + 1]

        out[2 * i] = x_even * c - x_odd * s
        out[2 * i + 1] = x_even * s + x_odd * c

    return out


def rope_matrix_python(xs, base=10000.0):
    out = []
    for pos, row in enumerate(xs):
        out.append(rope_one_vector_python(row, pos, base=base))
    return out


def rope_pytorch(x, positions=None, base=10000.0):
    assert x.ndim == 2
    seq_len, dim = x.shape
    assert dim % 2 == 0

    device = x.device
    dtype = x.dtype

    if positions is None:
        positions = torch.arange(seq_len, device=device, dtype=dtype)
    else:
        positions = positions.to(device=device, dtype=dtype)

    pair_idx = torch.arange(dim // 2, device=device, dtype=dtype)
    inv_freq = base ** (-2.0 * pair_idx / dim)
    thetas = positions[:, None] * inv_freq[None, :]

    cos_t = torch.cos(thetas)
    sin_t = torch.sin(thetas)

    x_even = x[:, 0::2]
    x_odd = x[:, 1::2]

    out_even = x_even * cos_t - x_odd * sin_t
    out_odd = x_even * sin_t + x_odd * cos_t

    out = torch.empty_like(x)
    out[:, 0::2] = out_even
    out[:, 1::2] = out_odd
    return out


if __name__ == "__main__":
    xs = [
        [1.0, 2.0, 3.0, 4.0],
        [0.5, -1.0, 2.0, 1.5],
        [3.2, 0.1, -0.7, 5.0],
    ]

    py_out = rope_matrix_python(xs)

    x_torch = torch.tensor(xs, dtype=torch.float64)
    torch_out = rope_pytorch(x_torch).tolist()

    print("python out:")
    for row in py_out:
        print(row)

    print("\ntorch out:")
    for row in torch_out:
        print(row)

    # چک اختلاف
    max_diff = 0.0
    for row_py, row_t in zip(py_out, torch_out):
        for a, b in zip(row_py, row_t):
            max_diff = max(max_diff, abs(a - b))

    print("\nmax abs diff =", max_diff)

python out:
[1.0, 2.0, 3.0, 4.0]
[1.1116221377419664, -0.11956681346419151, 1.9849002508320805, 1.5199246672933313]
[-1.4225996196334239, 2.8681370821874674, -0.7998533381332698, 4.985000966647556]

torch out:
[1.0, 2.0, 3.0, 4.0]
[1.1116221377419664, -0.11956681346419151, 1.9849002508320805, 1.5199246672933313]
[-1.4225996196334239, 2.8681370821874674, -0.7998533381332698, 4.985000966647556]

max abs diff = 0.0


In [9]:
from google.colab import drive
drive.mount("/content/drive")

from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/drive/MyDrive/Netra/netra_tokenizer.json"
tok = Tokenizer.from_file(TOKENIZER_PATH)
VOCAB_SIZE = tok.get_vocab_size()
print(f"Loaded tokenizer — vocab size: {VOCAB_SIZE:,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded tokenizer — vocab size: 32,000


In [13]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size: int = 32_000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_head: int = 64          # per-head dimension (d_model // n_heads)
    d_kv_latent: int = 512    # MLA compressed KV dimension
    d_q_latent: int = 768     # MLA compressed Q dimension
    d_rope: int = 32          # RoPE subspace dim per head
    ffn_hidden: int = 2048    # SwiGLU intermediate size
    max_seq_len: int = 2048
    norm_eps: float = 1e-6
    dropout: float = 0.0

    # MoE
    n_experts: int = 8        # total routed experts
    n_active_experts: int = 2 # top-k experts selected per token
    has_shared_expert: bool = True
    bias_update_speed: float = 0.001  # step size for bias-based load balancing

    @property
    def d_nope(self) -> int:
        """Non-positional part of each head (total head dim minus RoPE dim)."""
        return self.d_head - self.d_rope

cfg = ModelConfig(vocab_size=VOCAB_SIZE)
print(cfg)

NameError: name 'VOCAB_SIZE' is not defined

In [37]:
import torch
import numpy as np

indices = torch.arange(0, 768, 2).float()

max_len = 8
d_model = 4
position = np.arange(max_len)[:, np.newaxis]
div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

pe = np.zeros((max_len, d_model))
print(position * div_term)

pe[:, 0::2] = np.sin(position * div_term)  # Even indices
pe[:, 1::2] = np.cos(position * div_term)  # Odd indices



[[0.   0.  ]
 [1.   0.01]
 [2.   0.02]
 [3.   0.03]
 [4.   0.04]
 [5.   0.05]
 [6.   0.06]
 [7.   0.07]]


In [11]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

In [ ]:
class RotaryEmbedding(nn.Module):
    """Precomputes and caches the RoPE sin/cos frequency table."""

    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        # it's same as another line below but just more readable
        pair_idx = torch.arange(0, dim // 2, dtype=torch.float32)
        inv_freq = base ** (-2.0 * pair_idx / dim)
        # inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        t = torch.arange(max_seq_len, dtype=torch.float32)
        # freqs = torch.outer(t, inv_freq)
        freqs = t[:, None] * inv_freq[None, :] # convert t shape to (max_seq_len, 1) and inv_freq shape to (1, dim/2)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)

    def forward(self, seq_len: int):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply RoPE to x of shape (batch, n_heads, seq_len, d_rope).
    cos/sin have shape (seq_len, d_rope // 2).
    """
    # d = x.shape[-1]
    # x1 = x[..., : d // 2]
    # x2 = x[..., d // 2 :]
    x1 = x[:, 0::2]   # [seq_len, dim//2]
    x2 = x[:, 1::2]   

    cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d_rope//2)
    sin = sin.unsqueeze(0).unsqueeze(0)

    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

In [ ]:
import torch.nn.functional as F
import math

class MultiHeadLatentAttention(nn.Module):
    """
    Multi-head Latent Attention (MLA) from DeepSeek V2/V3.

    Q path:  hidden -> W_dq (compress to d_q_latent) -> W_uq (expand to n_heads * d_nope)
                                                      + W_qr (expand to n_heads * d_rope, then RoPE)
    KV path: hidden -> W_dkv (compress to d_kv_latent)  [<-- this is what gets cached]
             latent -> W_uk (expand to n_heads * d_nope)
                    + W_kr (expand to n_heads * d_rope, then RoPE)
             latent -> W_uv (expand to n_heads * d_head)
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.d_head = config.d_head
        self.d_nope = config.d_nope
        self.d_rope = config.d_rope
        self.d_kv_latent = config.d_kv_latent
        self.scale = 1.0 / math.sqrt(self.d_head)

        # --- Q path ---
        self.W_dq = nn.Linear(config.d_model, config.d_q_latent, bias=False)
        self.q_norm = RMSNorm(config.d_q_latent, eps=config.norm_eps)
        self.W_uq = nn.Linear(config.d_q_latent, config.n_heads * config.d_nope, bias=False)
        self.W_qr = nn.Linear(config.d_q_latent, config.n_heads * config.d_rope, bias=False)

        # --- KV path ---
        self.W_dkv = nn.Linear(config.d_model, config.d_kv_latent, bias=False)
        self.kv_norm = RMSNorm(config.d_kv_latent, eps=config.norm_eps)
        self.W_uk = nn.Linear(config.d_kv_latent, config.n_heads * config.d_nope, bias=False)
        self.W_kr = nn.Linear(config.d_kv_latent, config.n_heads * config.d_rope, bias=False)
        self.W_uv = nn.Linear(config.d_kv_latent, config.n_heads * config.d_head, bias=False)

        # --- Output ---
        self.W_o = nn.Linear(config.n_heads * config.d_head, config.d_model, bias=False)

    def forward(self, x: torch.Tensor, rope_cos: torch.Tensor, rope_sin: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape

        # Q: compress -> expand -> split nope + rope portions
        q_latent = self.q_norm(self.W_dq(x))
        q_nope = self.W_uq(q_latent).view(B, S, self.n_heads, self.d_nope).transpose(1, 2)
        q_rope = self.W_qr(q_latent).view(B, S, self.n_heads, self.d_rope).transpose(1, 2)
        q_rope = apply_rotary_emb(q_rope, rope_cos, rope_sin)

        # KV: compress -> expand
        kv_latent = self.kv_norm(self.W_dkv(x))
        k_nope = self.W_uk(kv_latent).view(B, S, self.n_heads, self.d_nope).transpose(1, 2)
        k_rope = self.W_kr(kv_latent).view(B, S, self.n_heads, self.d_rope).transpose(1, 2)
        k_rope = apply_rotary_emb(k_rope, rope_cos, rope_sin)
        v = self.W_uv(kv_latent).view(B, S, self.n_heads, self.d_head).transpose(1, 2)

        # Reassemble full Q and K by concatenating nope + rope parts
        q = torch.cat([q_nope, q_rope], dim=-1)  # (B, H, S, d_head)
        k = torch.cat([k_nope, k_rope], dim=-1)  # (B, H, S, d_head)

        # Causal attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        causal_mask = torch.triu(
            torch.full((S, S), float("-inf"), device=x.device), diagonal=1
        )
        attn_weights = attn_weights + causal_mask
        attn_weights = F.softmax(attn_weights, dim=-1)

        out = torch.matmul(attn_weights, v)                   # (B, H, S, d_head)
        out = out.transpose(1, 2).contiguous().view(B, S, -1) # (B, S, H*d_head)
        return self.W_o(out)

In [ ]:
class SwiGLUFFN(nn.Module):
    """Feed-forward network with SwiGLU gating (DeepSeek / LLaMA style)."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.w_gate = nn.Linear(config.d_model, config.ffn_hidden, bias=False)
        self.w_up   = nn.Linear(config.d_model, config.ffn_hidden, bias=False)
        self.w_down = nn.Linear(config.ffn_hidden, config.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
class MoELayer(nn.Module):
    """
    Mixture-of-Experts with auxiliary-loss-free load balancing (DeepSeek V3).

    Instead of adding a load-balancing loss to the training objective, each
    expert carries a bias term that is used ONLY for top-k gating decisions
    (not for computing combination weights). The bias is adjusted at each
    forward pass: experts receiving too many tokens get their bias decreased,
    underloaded experts get it increased. This steers routing toward balance
    without polluting the LM loss.
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_experts = config.n_experts
        self.n_active = config.n_active_experts
        self.bias_update_speed = config.bias_update_speed

        self.gate = nn.Linear(config.d_model, config.n_experts, bias=False)

        # Expert bias -- NOT a learned parameter (no gradients), updated by heuristic
        self.register_buffer("expert_bias", torch.zeros(config.n_experts))

        self.experts = nn.ModuleList(
            [SwiGLUFFN(config) for _ in range(config.n_experts)]
        )

        self.shared_expert = SwiGLUFFN(config) if config.has_shared_expert else None

    def forward(self, x: torch.Tensor):
        B, S, D = x.shape
        flat_x = x.view(-1, D)            # (B*S, D)
        N = flat_x.shape[0]

        router_logits = self.gate(flat_x)  # (B*S, n_experts)

        # Combination weights from raw logits (no bias)
        router_probs = F.softmax(router_logits, dim=-1)

        # Top-k selection uses logits + bias (bias steers routing, not weights)
        biased_logits = router_logits + self.expert_bias
        _, top_indices = torch.topk(biased_logits, self.n_active, dim=-1)

        # Gather the weights for chosen experts from the unbiased probs
        top_weights = router_probs.gather(dim=-1, index=top_indices)
        top_weights = top_weights / top_weights.sum(dim=-1, keepdim=True)

        # Dispatch tokens to experts
        out = torch.zeros_like(flat_x)
        for i in range(self.n_active):
            expert_idx = top_indices[:, i]
            weight = top_weights[:, i]

            for e in range(self.n_experts):
                mask = expert_idx == e
                if mask.any():
                    expert_out = self.experts[e](flat_x[mask])
                    out[mask] += weight[mask].unsqueeze(-1) * expert_out

        if self.shared_expert is not None:
            out = out + self.shared_expert(flat_x)

        out = out.view(B, S, D)

        # Update bias (no gradients, pure heuristic)
        if self.training:
            with torch.no_grad():
                tokens_per_expert = torch.zeros(self.n_experts, device=x.device)
                for i in range(self.n_active):
                    tokens_per_expert.scatter_add_(
                        0, top_indices[:, i], torch.ones(N, device=x.device)
                    )
                target_load = N * self.n_active / self.n_experts
                load_error = tokens_per_expert - target_load
                self.expert_bias -= self.bias_update_speed * load_error.sign()

        return out

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)
        self.attn = MultiHeadLatentAttention(config)
        self.ffn_norm = RMSNorm(config.d_model, eps=config.norm_eps)
        self.moe = MoELayer(config)

    def forward(self, x: torch.Tensor, rope_cos: torch.Tensor, rope_sin: torch.Tensor):
        x = x + self.attn(self.attn_norm(x), rope_cos, rope_sin)
        x = x + self.moe(self.ffn_norm(x))
        return x

In [ ]:
class Netra(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.rotary = RotaryEmbedding(config.d_rope, config.max_seq_len)
        self.layers = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.norm = RMSNorm(config.d_model, eps=config.norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None):
        """
        Returns:
            logits: (batch, seq_len, vocab_size)
            loss:   scalar cross-entropy loss, or None
        """
        B, S = input_ids.shape
        x = self.tok_emb(input_ids)

        rope_cos, rope_sin = self.rotary(S)

        for layer in self.layers:
            x = layer(x, rope_cos, rope_sin)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Netra(cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

expert_params_per_layer = sum(p.numel() for p in model.layers[0].moe.experts.parameters())
shared_params_per_layer = sum(p.numel() for p in model.layers[0].moe.shared_expert.parameters()) if cfg.has_shared_expert else 0
active_expert_params = (expert_params_per_layer // cfg.n_experts) * cfg.n_active_experts
active_ffn_per_layer = active_expert_params + shared_params_per_layer

print(f"Total parameters:       {total_params:>12,}")
print(f"Trainable parameters:   {trainable_params:>12,}")
print(f"Approx size (fp32):     {total_params * 4 / 1e6:.1f} MB")
print(f"Device:                 {device}")
print(f"\nMoE breakdown (per layer):")
print(f"  Experts:              {cfg.n_experts} total, {cfg.n_active_experts} active per token")
print(f"  Shared expert:        {'yes' if cfg.has_shared_expert else 'no'}")
print(f"  Expert params (all):  {expert_params_per_layer:,}")
print(f"  Active FFN params:    {active_ffn_per_layer:,}")

# Dummy forward pass
dummy_ids = torch.randint(0, cfg.vocab_size, (2, 128), device=device)
logits, loss = model(dummy_ids, targets=dummy_ids)
print(f"\nForward pass OK")
print(f"  Input shape:  {dummy_ids.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Loss:         {loss.item():.4f}")